<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 2 · Cash Flows, Discounting, and Present Value

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch02_cash_flows_discounting_present_value.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Everything in fixed income reduces to dated cash flows and the rates that discount them. Chapter 2 turns that principle into working machinery: payment timelines, discount factors under simple, annual, and continuous compounding, and prices read as sums of present-value contributions. You learn why a quoted rate is incomplete without its compounding convention, and why the final principal payment usually dominates a bond's value.

Afterwards you can value a small cash-flow schedule in Python, verify each payment's contribution to the price, and state how the total moves when the discount rate changes.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script loads the frozen three-payment teaching snapshot `ch02_cashflows.csv` and discounts it at a 4% continuously compounded annual rate (0.04 per year), printing the per-payment discount factors, present-value contributions, and the total price. It also compares the two-year discount factor under simple, annual, and continuous compounding at the same 4% rate. Check that the printed price is exactly the sum of the printed contributions.

In [ ]:
SCRIPT = ROOT / "code" / "ch02_cash_flows_discounting.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Read the printed table before quoting any single number.

- Confirm the price 101.3833 equals the sum of the three pv_part contributions (2.4505, 2.4020, 96.5309): the principal payment dominates value, not the coupons.
- Verify that discount factors decline with payment time (0.980199 to 0.941765) under the 4% continuous rate, 0.04 per year, and that payment times are stated in years.
- In the compounding comparison at T = 2 years, note the ordering simple (0.925926) > annual (0.924556) > continuous (0.923116): compounding the same 4% rate more often leaves less present value.
- Remember that the cash-flow file is an illustrative teaching snapshot, not market quotes, so the printed price is a convention-dependent teaching value.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch02_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Rebuild the present-value contribution table and add a one-basis-point sensitivity check. The result should make both value contribution and rate sensitivity visible by payment date.

In [ ]:
import numpy as np

flows = pd.read_csv(DATA / "ch02_cashflows.csv")
rate = 0.04
flows["discount"] = np.exp(-rate * flows.time)
flows["pv_part"] = flows.cashflow * flows.discount
flows["pv_part_bumped"] = (
    flows.cashflow * np.exp(-(rate + 0.0001) * flows.time)
)
flows["dv01_part"] = flows.pv_part - flows.pv_part_bumped
flows.round(6)

## Mini-lab: Rebuild the price from discount factors

The chapter's backbone identity is that a price is nothing more than the sum of discounted payments. The mini-lab rebuilds the chapter bond's three payments (2.5 at 0.5 and 1.0 years, 102.5 at 1.5 years) as in-notebook arrays and discounts them at a 4% continuously compounded annual rate, 0.04 per year, so each payment's contribution to the price is visible. All inputs are illustrative.

In [ ]:
import numpy as np
import pandas as pd

times = np.array([0.5, 1.0, 1.5])  # payment times in years
flows = np.array([2.5, 2.5, 102.5])  # coupons plus principal
rate = 0.04  # continuously compounded annual rate
pv = pd.DataFrame({"time": times, "cashflow": flows})
pv["discount"] = np.exp(-rate * pv.time)  # discount factors
pv["pv_part"] = pv.cashflow * pv.discount  # PV contributions
price = float(pv.pv_part.sum())  # price = sum of contributions
print(f"price: {price:.4f}")
pv.round(6)

### What to observe

With these inputs the calculation yields discount factors 0.980199, 0.960789, and 0.941765, contributions 2.450497, 2.401974, and 96.530865, and a price of 101.3833, exactly the sum of the contributions and matching the chapter script's output. The 102.5 principal-plus-coupon payment supplies roughly 95% of the value, which is why it also dominates the bond's rate sensitivity.

## Exercises

Use the mini-lab inputs and the chapter script to complete these checks, recording the compounding convention with every result.

1. **Bump the rate.** Recompute the price at 0.0425 (25 basis points higher) and confirm it falls to about 101.013. *Hint:* Only `rate` changes; the 1.5-year payment loses the most value.
2. **Switch the compounding basis.** Re-discount the same payments with annual compounding at the same 4% and compare the price with 101.3833. *Hint:* Use `(1 + rate) ** (-t)` per payment and state which convention produced each number.
3. **Strip a payment.** Drop the 1.0-year coupon and quantify how much price is lost. *Hint:* The price falls to about 98.9814, a small loss because the principal payment dominates value.
4. **Check the discount bounds.** Verify that every discount factor lies in (0, 1] for these non-negative times and rates. *Hint:* `np.exp(-rate * t)` cannot exceed one under these inputs.
5. **Trace the script.** Read `code/ch02_cash_flows_discounting.py` and confirm its printed price uses the same payments and continuous-compounding convention as the mini-lab. *Hint:* The script loads `ch02_cashflows.csv`, which holds the same three rows used here.

For more detail, see Chapter 2 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">